In [7]:
%pip install transformers datasets faiss-cpu scikit-learn
%pip install torch

In [10]:
from transformers import DPRContextEncoder, DPRContextEncoderTokenizer
import torch

import numpy as np
import random
from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

# Reading and preprocessing the data

In [11]:
filename = 'companyPolicies.txt'

In [12]:
def read_and_split(filename):
  with open(filename, encoding='utf-8') as file:
    text = file.read()

  paragraphs=text.split('\n')
  paragraphs=[para.strip() for para in paragraphs if len(para.strip())>0]
  return paragraphs

paragraphs = read_and_split(filename)

for i in range(3):
  print(f"Sample: {i+1}\nparagraph: {paragraphs[i]}\n")

Sample: 1
paragraph: 1.	Code of Conduct

Sample: 2
paragraph: Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.

Sample: 3
paragraph: Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.



# Encoding texts into embeddings

## **1. Tokenization**

In [13]:
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

### sample

In [16]:
text = [("How are you?", "I am fine."), ("What's up?", "Not much.")]

In [17]:
token_info=context_tokenizer(text, return_tensors='pt', padding=True,
                             truncation=True, max_length=256)
token_info

{'input_ids': tensor([[ 101, 2129, 2024, 2017, 1029,  102, 1045, 2572, 2986, 1012,  102],
        [ 101, 2054, 1005, 1055, 2039, 1029,  102, 2025, 2172, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [19]:
for s in token_info['input_ids']:
  print(context_tokenizer.convert_ids_to_tokens(s))

['[CLS]', 'how', 'are', 'you', '?', '[SEP]', 'i', 'am', 'fine', '.', '[SEP]']
['[CLS]', 'what', "'", 's', 'up', '?', '[SEP]', 'not', 'much', '.', '[SEP]']


# **2. Encoding**

In [20]:
context_encoder = DPRContextEncoder.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')

config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] DPRContextEncoder LOAD REPORT from: facebook/dpr-ctx_encoder-single-nq-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
ctx_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
ctx_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
tokens=context_tokenizer( paragraphs[:20], return_tensors='pt', padding=True,
                         truncation=True, max_length=256)
tokens['input_ids']

tensor([[  101,  1015,  1012,  ...,     0,     0,     0],
        [  101,  2256,  3642,  ...,     0,     0,     0],
        [  101, 11109,  1024,  ...,     0,     0,     0],
        ...,
        [  101,  2256, 15680,  ..., 15680,  1012,   102],
        [  101,  1017,  1012,  ...,     0,     0,     0],
        [  101,  2256,  4274,  ...,     0,     0,     0]])

In [27]:
outputs=context_encoder(**tokens)
outputs.pooler_output

tensor([[ 0.9058,  0.1277, -0.0524,  ..., -0.3109, -0.0634,  0.1343],
        [ 0.8406,  0.6897,  0.0202,  ..., -0.4066,  0.0511,  0.3425],
        [ 0.8745,  0.6094, -0.1193,  ..., -0.0681, -0.1915,  0.0144],
        ...,
        [ 0.6020,  0.3124,  0.0835,  ..., -0.5222, -0.2321,  0.2006],
        [ 0.1947, -0.1456,  0.1621,  ...,  0.0998,  0.1219,  0.4370],
        [ 0.3869,  0.4322,  0.3379,  ..., -0.4071, -0.0797, -0.0184]],
       grad_fn=<SelectBackward0>)

## 3. Aggregation

In [28]:
embeddings=[]
for text in paragraphs[0:5]:
    inputs = context_tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=256)
    outputs = context_encoder(**inputs)
    embeddings.append(outputs.pooler_output)
    print("number of samples:")
    print(len(embeddings))
    print(" samples shape:")
    print(outputs.pooler_output.shape)

number of samples:
1
 samples shape:
torch.Size([1, 768])
number of samples:
2
 samples shape:
torch.Size([1, 768])
number of samples:
3
 samples shape:
torch.Size([1, 768])
number of samples:
4
 samples shape:
torch.Size([1, 768])
number of samples:
5
 samples shape:
torch.Size([1, 768])


In [29]:
torch.cat(embeddings).detach().numpy().shape

(5, 768)

# All together

In [30]:
def encode_contexts(text_list):
  embeddings=[]
  for text in text_list:
    tokens=context_tokenizer(text, return_tensors='pt', padding=True,
                             truncation=True, max_length=256)
    outputs=context_encoder(**tokens)
    embeddings.append(outputs.pooler_output)
  return torch.cat(embeddings).detach().numpy()

context_embeddings = encode_contexts(paragraphs)

array([[ 0.90580297,  0.1276515 , -0.05235763, ..., -0.31091243,
        -0.06342757,  0.13434964],
       [ 0.840582  ,  0.689665  ,  0.02017527, ..., -0.40660438,
         0.05106763,  0.342531  ],
       [ 0.87451184,  0.6094326 , -0.11934307, ..., -0.06810481,
        -0.19154234,  0.01444948],
       ...,
       [ 0.43880546,  0.11872502,  0.07763571, ..., -0.4157035 ,
        -0.5836465 , -0.07130916],
       [-0.16259827,  0.02009357,  0.3065368 , ..., -0.3257994 ,
        -0.6220858 ,  0.07232004],
       [ 0.15627459,  0.2912436 ,  0.26207903, ..., -0.358435  ,
        -0.34246358,  0.17852928]], dtype=float32)